<a href="https://colab.research.google.com/github/BF667-IDLE/rvc-batch/blob/main/rvc_batch_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RVC-Batch: Voice Conversion Demo

A simple, high-quality voice conversion tool focused on ease of use and batch processing.

**Features:**
- Single file & batch folder inference
- Multiple F0 methods (RMVPE, CREPE, FCPE, SWIPE, etc.)
- RVC v1 & v2 model support
- CUDA / MPS / CPU support

## 1. Installation

Install dependencies and clone the repository.

In [ ]:
#@title Clone repo & install dependencies
!git clone https://github.com/BF667-IDLE/rvc-batch.git
%cd rvc-batch
!uv pip install -r requirements.txt

## 2. Download Models

Download the HuBERT base model and your preferred F0 predictor.

In [ ]:
#@title Download required models
import os
os.makedirs("models", exist_ok=True)

from main.utils import check_predictors, check_embedders

f0_method = "rmvpe"  # @param ["rmvpe", "crepe-full", "crepe-tiny", "crepe-small", "crepe-medium", "crepe-large", "fcpe", "swipe"]

print(f"Downloading HuBERT model...")
check_embedders()

print(f"Downloading {f0_method} predictor...")
check_predictors(f0_method)

print("All models downloaded!")

## 3. Load Your Voice Model

Upload your RVC voice model (`.pth`) and optionally an index file (`.index`).
Or provide a Google Drive / direct download link.

In [ ]:

#@title Download voice model from URL

import os, zipfile, shutil, urllib.request, io
from google.colab import files

# @markdown Enter the URL and model name:
url = "https://huggingface.co/marup/ApplejackRVC/resolve/main/Applejack.zip"  # @param {type:"string"}
model_name = "Applejack"  # @param {type:"string"}
rvc_models_dir = "/content/rvc-batch/models"  # @param {type:"string"}

def extract_zip(extraction_folder, zip_name):
    os.makedirs(extraction_folder, exist_ok=True)
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall(extraction_folder)
    if os.path.exists(zip_name):
        os.remove(zip_name)
    index_filepath, model_filepath = None, None
    for root, dirs, files in os.walk(extraction_folder):
        for name in files:
            file_path = os.path.join(root, name)
            file_size = os.stat(file_path).st_size
            if name.endswith('.index') and file_size > 1024 * 100:
                index_filepath = file_path
            if name.endswith('.pth') and file_size > 1024 * 1024 * 40:
                model_filepath = file_path
    if not model_filepath:
        print(f"Error: no .pth model found in zip.")
        return False
    target_model = os.path.join(extraction_folder, os.path.basename(model_filepath))
    if model_filepath != target_model:
        shutil.move(model_filepath, target_model)
    if index_filepath:
        target_index = os.path.join(extraction_folder, os.path.basename(index_filepath))
        if index_filepath != target_index:
            shutil.move(index_filepath, target_index)
    for item in os.listdir(extraction_folder):
        item_path = os.path.join(extraction_folder, item)
        if os.path.isdir(item_path):
            shutil.rmtree(item_path)
    return True

def dl_model(url, model_name):
    try:
        if not url or not model_name:
            print("Please provide both URL and model name.")
            return
        extraction_folder = os.path.join(rvc_models_dir, model_name)
        if os.path.exists(extraction_folder):
            print(f"Model '{model_name}' already exists, skipping.")
            return
        if 'pixeldrain.com' in url:
            file_id = url.split('/')[-1]
            if '/api/file/' not in url:
                url = f'https://pixeldrain.com/api/file/{file_id}'
        zip_name = f"{model_name}.zip"
        urllib.request.urlretrieve(url, zip_name)
        if extract_zip(extraction_folder, zip_name):
            files = [f for f in os.listdir(extraction_folder) if os.path.isfile(os.path.join(extraction_folder, f))]
            print(f"Done: {model_name} ({', '.join(files)})")
        else:
            print("Failed to extract model files.")
    except Exception as e:
        print(f"Error: {e}")
        if 'zip_name' in locals() and os.path.exists(zip_name):
            os.remove(zip_name)

upload_option = "URL"  # @param ["URL", "Upload from computer"]
if upload_option == "Upload from computer":
    uploaded = files.upload()
    for filename in uploaded.keys():
        if filename.endswith('.zip'):
            model_name = filename.replace('.zip', '')
            extraction_folder = os.path.join(rvc_models_dir, model_name)
            if extract_zip(extraction_folder, filename):
                print(f"Done: {model_name}")
            break
else:
    dl_model(url=url, model_name=model_name)

In [ ]:
#@title Initialize models
import torch
import os

from main.infer.infer import Config, load_hubert, get_vc

# Voice model paths (adjust after downloading in step 3)
voice_path = "/content/rvc-batch/models/Applejack/MLP05Applejack_e80_s1600.pth"  # @param {type:"string"}
index_path = "/content/rvc-batch/models/Applejack/added_IVF390_Flat_nprobe_1_MLP05Applejack_v2.index"  # @param {type:"string"}

# Auto-detect device
if torch.cuda.is_available():
    device = "cuda:0"
    is_half = True
    print(f"GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = "mps"
    is_half = True
    print("Using Apple Silicon (MPS)")
else:
    device = "cpu"
    is_half = True
    print("Using CPU")

config = Config(device, is_half)

print("Loading HuBERT model...")
hubert_model = load_hubert(device, is_half, "models/hubert_base.pt")

print(f"Loading voice model from {voice_path}...")
cpt, version, net_g, tgt_sr, vc = get_vc(device, is_half, config, voice_path)

print(f"Model version: {version}")
print(f"Target sample rate: {tgt_sr}Hz")
print(f"Index file: {index_path if index_path and os.path.exists(index_path) else '(none)'}")
print("Ready for inference!")

## 4. Single File Inference

Convert a single audio file with full parameter control.

In [ ]:

#@title Single file inference
import os
from google.colab import files
from main.infer.infer import rvc_infer

# Parameters
input_file = ""  # @param {type:"string"}
output_name = "output.wav"  # @param {type:"string"}
pitch_change = 0  # @param {type:"slider", min:-12, max:12, step:1}
f0_method = "rmvpe"  # @param ["rmvpe", "crepe-full", "crepe-tiny", "crepe-small", "crepe-medium", "crepe-large", "fcpe", "swipe", "pm", "harvest", "dio", "yin", "pyin"]
index_rate = 0.75  # @param {type:"slider", min:0, max:1, step:0.05}
filter_radius = 3  # @param {type:"integer"}
rms_mix_rate = 0.5  # @param {type:"slider", min:0, max:1, step:0.05}
protect = 0.33  # @param {type:"slider", min:0, max:0.5, step:0.01}

os.makedirs("output", exist_ok=True)
output_path = f"output/{output_name}"

print(f"\nConverting {input_file}...")
print(f"  Pitch: {pitch_change:+d} st | F0: {f0_method} | Index rate: {index_rate}")

rvc_infer(
    index_path=index_path,
    index_rate=index_rate,
    input_path=input_file,
    output_path=output_path,
    pitch_change=pitch_change,
    f0_method=f0_method,
    cpt=cpt,
    version=version,
    net_g=net_g,
    filter_radius=filter_radius,
    tgt_sr=tgt_sr,
    rms_mix_rate=rms_mix_rate,
    protect=protect,
    crepe_hop_length=128,
    vc=vc,
    hubert_model=hubert_model,
)

print(f"\nDone! Output saved to {output_path}")
files.download(output_path)

## 5. Batch Folder Inference

Process an entire folder of audio files at once.
Supports: `.wav`, `.mp3`, `.flac`, `.ogg`, `.opus`, `.m4a`, `.wma`, `.aac`, `.webm`

In [ ]:
#@title Batch folder inference
import os
import shutil
from google.colab import files
from main.infer.infer import rvc_infer_batch

# Parameters
input_folder = "input_batch"  # @param {type:"string"}
output_folder = "output_batch"  # @param {type:"string"}
pitch_change = 0  # @param {type:"slider", min:-12, max:12, step:1}
f0_method = "rmvpe"  # @param ["rmvpe", "crepe-full", "crepe-tiny", "crepe-small", "crepe-medium", "crepe-large", "fcpe", "swipe", "pm", "harvest", "dio", "yin", "pyin"]
index_rate = 0.75  # @param {type:"slider", min:0, max:1, step:0.05}
filter_radius = 3  # @param {type:"integer"}
rms_mix_rate = 0.5  # @param {type:"slider", min:0, max:1, step:0.05}
protect = 0.33  # @param {type:"slider", min:0, max:0.5, step:0.01}

# Upload a ZIP file or multiple audio files
print("Upload audio files or a ZIP archive:")
uploaded = files.upload()

os.makedirs(input_folder, exist_ok=True)
os.makedirs(output_folder, exist_ok=True)

for fname, data in uploaded.items():
    fpath = os.path.join(input_folder, fname)
    with open(fpath, "wb") as f:
        f.write(data)
    print(f"  Saved: {fname}")

# Extract ZIP if uploaded
for fname in os.listdir(input_folder):
    if fname.endswith(".zip"):
        print(f"\nExtracting {fname}...")
        shutil.unpack_archive(os.path.join(input_folder, fname), input_folder)
        os.remove(os.path.join(input_folder, fname))

# Count files
audio_extensions = {".wav", ".mp3", ".flac", ".ogg", ".opus", ".m4a", ".wma", ".aac", ".webm"}
file_count = sum(1 for f in os.listdir(input_folder) if os.path.splitext(f)[1].lower() in audio_extensions)

print(f"\n{'='*50}")
print(f"Batch inference: {file_count} file(s)")
print(f"  Pitch: {pitch_change:+d} st | F0: {f0_method} | Index rate: {index_rate}")
print(f"{'='*50}")

summary = rvc_infer_batch(
    index_path=index_path,
    index_rate=index_rate,
    input_path=input_folder,
    output_path=output_folder,
    pitch_change=pitch_change,
    f0_method=f0_method,
    cpt=cpt,
    version=version,
    net_g=net_g,
    filter_radius=filter_radius,
    tgt_sr=tgt_sr,
    rms_mix_rate=rms_mix_rate,
    protect=protect,
    crepe_hop_length=128,
    vc=vc,
    hubert_model=hubert_model,
)

print(f"\nResults: {summary['processed']} processed, {summary['failed']} failed, {summary['skipped']} skipped")

# Download all results as ZIP
zip_path = "output_batch.zip"
shutil.make_archive("output_batch", "zip", output_folder)
files.download(zip_path)

## 6. Playback Results

Listen to converted files directly in Colab.

In [ ]:
#@title Listen to output files
import glob
from IPython.display import Audio, display

output_dir = "output"  # @param ["output", "output_batch"]

wav_files = sorted(glob.glob(os.path.join(output_dir, "*.wav")))

if not wav_files:
    print(f"No .wav files found in {output_dir}/")
else:
    print(f"Found {len(wav_files)} file(s):\n")
    for f in wav_files:
        print(f"  {os.path.basename(f)}")
        display(Audio(filename=f))